In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from utils.helpers import *
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

device = find_backend()

Currently using:  mps


In [3]:
import pandas as pd

splits = {'train': 'Personality Datasets - Reddit/train_set.csv', 'validation': 'Personality Datasets - Reddit/val_set.csv', 'test': 'Personality Datasets - Reddit/eval_set.csv'}
train = pd.read_csv("hf://datasets/Fatima0923/Automated-Personality-Prediction/" + splits["train"])
test = pd.read_csv("hf://datasets/Fatima0923/Automated-Personality-Prediction/" + splits["test"])
validation = pd.read_csv("hf://datasets/Fatima0923/Automated-Personality-Prediction/" + splits["validation"])

In [36]:
p_type_names = ['agreeableness', 'openness', 'conscientiousness','extraversion', 'neuroticism']
train["personality"] = train[p_type_names].apply(lambda x: torch.tensor(x), axis = 1)
validation["personality"] = validation[p_type_names].apply(lambda x: torch.tensor(x), axis = 1)
test["personality"] = test[p_type_names].apply(lambda x: torch.tensor(x), axis = 1)


/var/folders/l4/tfnrkckd1mj12_lf97sckpfc0000gn/T/ipykernel_35898/212605137.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  train["personality"] = train[p_type_names].apply(lambda x: torch.tensor(x), axis = 1)
/var/folders/l4/tfnrkckd1mj12_lf97sckpfc0000gn/T/ipykernel_35898/212605137.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  validation["personality"] = validation[p_type_names].apply(lambda x: torch.tensor(x), axis = 1)
/var/folders/l4/tfnrkckd1mj12_lf97sckpfc0000gn/T/ipykernel_35898/212605137.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer ke

In [39]:
from transformers import AutoTokenizer, DistilBertModel
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# teacher_model = DistilBertModel.from_pretrained("distilbert-base-uncased")

To use data.metrics please install scikit-learn. See https://scikit-learn.org/stable/index.html


In [41]:
from models import encoder_decoder, decoder_only

model_type = "encoder_decoder" 
# model_type = "decoder_only"

vocab_size = tokenizer.vocab_size
hidden_dim=128
num_heads=2
dim_feedforward=2048
num_layers_enc=2
num_layers_dec=2
dropout=0.1
max_length=128
p_tags=5
ignore_index = -1

if model_type == "encoder_decoder":
    model =  encoder_decoder.EncoderDecoder(vocab_size,device, hidden_dim, num_heads, dim_feedforward, num_layers_enc, num_layers_dec, dropout,
                                            max_length, p_tags, ignore_index)

model = model.to(device)

In [42]:
input_test = torch.tensor(tokenizer.encode("This is an example")).unsqueeze(0).to(device)
output_test = torch.tensor(tokenizer.encode("This is output example")).unsqueeze(0).to(device)
personality_test = torch.rand(1,5).to(device)

input_test, output_test, personality_test

(tensor([[2023, 2003, 2019, 2742]], device='mps:0'),
 tensor([[2023, 2003, 6434, 2742]], device='mps:0'),
 tensor([[0.7403, 0.0925, 0.7622, 0.5861, 0.1634]], device='mps:0'))

In [43]:
input_test

tensor([[2023, 2003, 2019, 2742]], device='mps:0')